# ebook2audiobook - Android APK build (Google Colab)

Buildozer only runs on Linux, so the APK for the thin Android client
(`android_client/`) is built here on a free Colab machine.

## How to use
1. Runtime type: plain **CPU** is enough (no GPU needed).
2. Run the two cells top to bottom. The first build takes ~30-50 min
   (Android SDK/NDK download); repeat builds are much faster.
3. The finished `ebook2audiobook-*.apk` downloads automatically.
4. On the phone (Android 10/11): allow installing from unknown sources,
   open the APK, done.

If the build fails, cell (2) prints the error lines from `build.log`
automatically - copy them when reporting a problem.

In [ ]:
#@title (1) Get the client sources & install buildozer
REPO = 'Tarkas/Book-to-audiobook'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

import os, subprocess, sys

cmds = [
    'sudo apt-get update -qq',
    # autotools/gettext family is needed to build libffi & friends
    'sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf automake autopoint '
    'libtool libtool-bin libltdl-dev gettext patch pkg-config ccache '
    'zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev',
    # cython 0.29.37 is the newest 0.29.x and supports Python 3.12 (current Colab)
    'pip install -q buildozer cython==0.29.37',
]
for c in cmds:
    print('\n$', c)
    r = subprocess.run(c, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'Setup command failed (exit {r.returncode}): {c}')

if not os.path.isdir('ebook2audiobook'):
    r = subprocess.run(
        f'git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git ebook2audiobook',
        shell=True)
    if r.returncode != 0:
        raise SystemExit('git clone failed - check REPO/BRANCH above')

%cd ebook2audiobook/android_client
print('\nSetup done. Python:', sys.version.split()[0])

In [ ]:
#@title (2) Build the APK & download
import glob, os, re, subprocess

# Gradle / Android toolchain needs Java 17 (Colab default may be 11)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# buildozer downloads the Android SDK/NDK on first run (long, one-off).
# Full output goes to build.log; only progress markers are shown here.
progress = re.compile(
    r'(-> running|# Prepar|# Build|# Download|# Install|# Unpack|# Compil|'
    r'BUILD (SUCCESSFUL|FAILED)|STDERR|[Ee]rror|FAILED|Exception)')
with open('build.log', 'w', errors='replace') as log:
    proc = subprocess.Popen('yes | buildozer -v android debug', shell=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors='replace')
    for line in proc.stdout:
        log.write(line)
        if progress.search(line):
            print(line, end='')
    proc.wait()
print('\nbuildozer exit code:', proc.returncode)

apks = sorted(glob.glob('bin/*.apk'))
if apks:
    apk = apks[-1]
    print('APK ready:', apk)
    try:
        from google.colab import files
        files.download(apk)
    except Exception as e:
        print('Auto-download failed:', e, '- grab the file from the Files panel:', apk)
else:
    lines = open('build.log', errors='replace').read().splitlines()
    print('\n--- BUILD FAILED, last 80 lines of build.log: ---')
    print('\n'.join(lines[-80:]))
    err = [l for l in lines if re.search(r'error:|Error:|FAILED|Exception|No such file|not found', l)]
    if err:
        print('\n--- lines mentioning errors (copy these when reporting): ---')
        print('\n'.join(err[-40:]))

In [ ]:
#@title (3) Diagnostics - real failures only + tail of build.log
import glob, re

apks = sorted(glob.glob('bin/*.apk'))
print('APK files:', apks if apks else 'NONE')

lines = open('build.log', errors='replace').read().splitlines()

# Real failure markers only. p4a's "Trying first build ... this is expected
# to fail" passes produce harmless clang/ccache errors - do not report them.
real = [i for i, l in enumerate(lines)
        if re.search(r'# Command failed|BUILD FAILED|Aborted!|buildozer.*[Ee]rror', l)]
if real:
    for i in real:
        lo, hi = max(0, i - 40), min(len(lines), i + 10)
        print('=' * 70)
        print('\n'.join(lines[lo:hi]))
    print('=' * 70)
    print('Copy everything above when reporting the problem.')
else:
    print('No real failure markers found.')

print('\n--- last 60 lines of build.log ---')
print('\n'.join(lines[-60:]))